# Step 08.1 — 为什么普通 MLP 处理序列不够自然？

**历史主线：MLP → RNN → LSTM/GRU → Seq2Seq → Attention → Transformer → GPT。**

Step 4.4 我们写过 `C → 4C → GELU → C` 的逐位置 MLP；Step 7 又得到 `(B,T,C)`。现在的问题是：**每个 token 的特征会变，但它怎样利用前面读过的 token？**

本讲只建立 RNN 出现的动机，正式的 RNN 权重方程、MLX RNN Cell、BPTT 留给 Step 8.2–9，避免提前堆叠复杂公式。

本讲内部五部分：① 序列任务的三个需求；② position-wise MLP 的边界；③ flatten + dense MLP 的代价；④ recurrent hidden state 的直觉；⑤ shape、计算图和 RNN 的代价预览。


## 1. 序列任务究竟比普通向量任务多了什么？

判断一张图像的类别时，一个典型 MLP 可以收到固定维度的输入向量。但文本、语音和时间序列不仅有数值，还包含**顺序（order）与历史（history）**，长度还可能变化。

看两个只改变排列的例子：

```text
dog chases cat
cat chases dog
```

它们包含同样三个词，却表达了不同事件。这里用词只是为了直观；我们的教学 MiniGPT 仍使用 character-level tokenizer。

对于 next-token prediction，我们真正希望：

```text
当前预测不仅依赖 x_t，也可能依赖 x_0 ... x_(t-1)
输入可以在不同样本间有不同的 T
相同的计算规则可以重复用于每一个新到来的 token
```

想象一段流式文本：先收到 `I`，再收到 `I love`，最后收到 `I love ML`。我们希望模型能随着新 token 到来更新对前文的表示，而不必每多一个 token 就重设计模型权重。

**关键区分：**“能读取一个序列”和“在结构上适合变长序列、共享跨时间计算”不是同一回事。MLP 并非数学上完全不能处理序列，下面区分两种用法。


## 2. 逐位置 MLP：会混合 C，但不会让 token 之间交流

回顾 Step 4.4 的 MLP：

```text
每个位置上的 C 维向量
   ↓ Linear(C,4C)
   ↓ GELU
   ↓ Linear(4C,C)
仍是该位置上的 C 维向量
```

当输入是 `(B,T,C)` 时，MLX 的 `nn.Linear` 默认只作用于**最后一个特征轴 C**，所有 `(b,t)` 位置共享相同的参数。

因此可以写成：

$$y_{b,t}=f_\theta(h_{b,t})$$

右侧**没有**其他位置的 `h_(b,s)`，所以这个子层自身不执行跨 token 信息交换。即使 `h` 已加过 position embedding，这个限制仍然存在。

做一个控制变量实验：只修改第 1 个位置，检查第 3 个位置的输出会不会变。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

class PositionMLP(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.fc = nn.Linear(C, 4*C, bias=False)
        self.gelu = nn.GELU()
        self.proj = nn.Linear(4*C, C, bias=False)

    def __call__(self, x):
        return self.proj(self.gelu(self.fc(x)))

C, T, V = 4, 4, 6
wte = nn.Embedding(V, C)
wpe = nn.Embedding(T, C)
mlp = PositionMLP(C)

ids_a = mx.array([[1, 2, 3, 4]])
ids_b = mx.array([[1, 0, 3, 4]])  # 只改 position 1

pos = wpe(mx.arange(T))
h_a = wte(ids_a) + pos
h_b = wte(ids_b) + pos

out_a = mlp(h_a)
out_b = mlp(h_b)
mx.eval(out_a, out_b)

print('output shape:', out_a.shape)
print('position 1 changed:', not bool(mx.allclose(out_a[:,1,:], out_b[:,1,:])))
print('position 3 unchanged:', bool(mx.allclose(out_a[:,3,:], out_b[:,3,:])))


这个实验的核心不是随机输出的具体数字，而是**依赖关系**：

```text
改变 x[1] → y[1] 可以改变
改变 x[1] → y[3] 不会因此改变（单独的逐位置 MLP）
```

所以单独使用逐位置 MLP 做 next-token prediction 时，位置 3 并不能通过这层知道位置 1 发生了什么。这个问题不是“参数不够多”，而是计算图里压根没有跨 T 的边。

需要修正一个常见误解：**MLP 本身并非永远不能跨 token 交流。**如果先把整段序列展平，再让 Dense 层连接全部位置，当然可以；但它会引出另一个困难。


## 3. 把整段序列 Flatten 再送进 MLP：能混合时间，却容易绑定固定长度

一种直接的替代方案是：把 `(B,T,C)` 展平成 `(B,T*C)`，再交给 `nn.Linear(T*C,H)`。

这时 Dense 层一个输出可以依赖任意输入位置，因此**确实能**混合序列中不同 token 的信息。

但其权重 shape 与 T 有关：

```text
T=4, C=4 → 输入是 16 维
T=6, C=4 → 输入是 24 维
```

一旦创建 `nn.Linear(16,H)`，原权重不能未经修改就接收 24 维输入。固定长度、padding/masking、复杂时间轴混合结构都可以缓解部分问题，但比“同一规则连续处理新 token”要笨拙。


In [ ]:
B, T, C, H = 1, 4, 4, 8
hidden = mx.arange(B*T*C, dtype=mx.float32).reshape(B,T,C)
flattened = hidden.reshape(B,T*C)
whole_sequence_mlp = nn.Linear(T*C, H, bias=False)
output = whole_sequence_mlp(flattened)

print('sequence shape:', hidden.shape)
print('flattened shape:', flattened.shape)
print('weight shape   :', whole_sequence_mlp.weight.shape)
print('output shape   :', output.shape)
print('if T=6, required input width:', 6*C)
print('this layer expects input width:', T*C)


### 用参数量趋势看“固定长度权重”的代价

若固定每步 feature width `C` 和输出 width `H`，单个 `Linear(T*C,H,bias=False)` 的权重数为：

$$P_{flat}=TCH$$

它随 T 线性增长。如果还要做 `T*C -> T*C` 的全序列 Dense，单层就会有 $(TC)^2$ 规模的权重。

为了对照，后面一个**经典简单 RNN cell** 常见的参数规模可写为：

$$P_{RNN}=HC+H^2+H$$

它依赖 C 和 hidden size H，**不直接依赖处理多少个时间步 T**，因为时间步间复用同一套权重。

下面图表**只比较“参数量如何随 T 改变”**，不代表两种模型表达能力、训练速度或最终准确率相等。


In [ ]:
import matplotlib.pyplot as plt

sequence_lengths = [2, 4, 8, 16, 32, 64]
C, H = 4, 8
flat_params = [T*C*H for T in sequence_lengths]
shared_rnn_params = [H*C + H*H + H for _ in sequence_lengths]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sequence_lengths, flat_params, marker='o', label='Flatten + Linear: T*C*H')
ax.plot(sequence_lengths, shared_rnn_params, marker='s', label='RNN cell: H*C + H² + H')
ax.set_xlabel('sequence length T')
ax.set_ylabel('parameter count')
ax.set_title('Dependence of parameter count on T (illustration only)')
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()


绘图小技巧：`ax.plot(x,y,marker=...)` 用于观察随某个变量改变的趋势；`ax.legend()` 让多个方法可辨认；`ax.grid()` 帮助读取纵轴。注意这张图比较的是**权重规模**，不是所有计算/内存成本。

**现实边界：**现代模型也可以用 CNN、temporal MLP、padding、attention 等处理序列。RNN 不是逻辑上唯一可行的方案；它提供了一种很自然的**参数共享 + 可递推历史状态**结构。


## 4. RNN 的关键想法：用同一规则维护会更新的 Hidden State

研究者可以采取另一条路线：不把整段变长历史一次展成固定宽度输入，而是维护一个**固定大小的状态向量** `h_t`，随着新 token 到来反复更新。

抽象写成：

$$h_t=f_\theta(x_t,h_{t-1})$$

含义：当前新输入 `x_t` + 上一步保存的状态 `h_(t-1)` → 新的历史摘要 `h_t`。最重要的是 **同一个 f 和同一套参数 θ 被所有时间步复用**。

先不学矩阵版 RNN。我们用一个没有 trainable parameter 的**教学用标量递推**来体验状态：

$$h_t=x_t+0.5h_{t-1},\quad h_{-1}=0$$

输入 `[1,0,2]`：

```text
读 1：h=1 + 0.5*0    = 1
读 0：h=0 + 0.5*1    = 0.5
读 2：h=2 + 0.5*0.5  = 2.25
```

相同元素倒过来 `[2,0,1]`，最终是 `1.5`。先后顺序影响了历史状态。


In [ ]:
def tiny_stream_state(sequence):
    h = mx.array(0.0)
    states = []
    for t in range(sequence.shape[0]):
        h = sequence[t] + 0.5*h
        states.append(h)
    return mx.stack(states)

forward = tiny_stream_state(mx.array([1.0, 0.0, 2.0]))
reverse = tiny_stream_state(mx.array([2.0, 0.0, 1.0]))
longer = tiny_stream_state(mx.array([1.0, 0.0, 2.0, 3.0]))
mx.eval(forward, reverse, longer)

print('order 1,0,2    :', forward)
print('order 2,0,1    :', reverse)
print('longer sequence:', longer)
print('same final state?', bool(mx.allclose(forward[-1],reverse[-1])))


这个玩具例子展示了 RNN 的三个动机：

一是**顺序敏感**：当前状态来自之前状态；二是**可连续处理**：长度从 3 变成 4 时，同一递推公式照样可以继续；三是**参数共享**：无需为每个时间位置创建一套新权重。

但请不要误认为固定维度 `h_t` 可以无限、无损地记下历史！它是压缩的、依赖任务学习的摘要，长程信息可能遗失。这个玩具里的 `0.5` 反复相乘，还预示了后面 Step 9 的梯度消失问题。

正式 RNN 将让 `x_t` 与 `h_t` 都是向量，并学习自己的权重；其精确方程和手算放到 **8.2**。


## 5. 三种计算路径对比：为什么 RNN 适合“流式”序列？

现在在形状上把问题拼起来：

```text
一批 token embeddings: X.shape=(B,T,C)
                                      ↓
position-wise MLP: 每个 (b,t) 独立 C -> C；不交换 T 信息
flatten + Dense:   (B,T,C)->(B,T*C)->(B,H)；输入 width 绑定 T
recurrent state:   每步 x_t=(B,C)，h_prev=(B,H)，产生 h_t=(B,H)
                   连续处理 T 步后可以收集 (B,T,H)
```

`H` 是 hidden-state width（不必等于输入 feature width C）。Batch 中各样本可以并行更新，但同一序列的 `h_t` 要等 `h_(t-1)`，这让时间轴 T 具有串行依赖。

下面用一个计算图对照帮助记忆。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

for ax in axes:
    ax.set_xlim(0, 8)
    ax.set_ylim(-0.2, 2.3)
    ax.axis('off')

def node(ax, x, y, label):
    ax.text(x, y, label, ha='center', va='center',
            bbox={'boxstyle':'round,pad=0.3', 'fill':False})

def edge(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops={'arrowstyle':'->'})

for i,x in enumerate([1,4,7]):
    node(axes[0], x, 1.7, f'x{i}')
    node(axes[0], x, 0.5, f'MLP(x{i})')
    edge(axes[0], x, 1.45, x, 0.78)
axes[0].set_title('Position-wise MLP: no cross-time path')

for i,x in enumerate([1,4,7]):
    node(axes[1], x, 1.7, f'x{i}')
    node(axes[1], x, 0.5, f'h{i}')
    edge(axes[1], x, 1.45, x, 0.78)
for left,right in [(1,4),(4,7)]:
    edge(axes[1], left+0.55, 0.5, right-0.55, 0.5)
axes[1].set_title('Recurrent state: history flows through h')

plt.show()


可视化方法：`plt.subplots(1,2)` 创建并排比较图；`ax.text(...,bbox=...)` 画节点；`ax.annotate(...,arrowprops=...)` 画依赖箭头；`ax.axis('off')` 隐藏不需要的坐标轴。计算图里箭头代表**信息依赖**，不是物理 GPU 核心或真实运行时间。

**最后保留两个重要代价：**

1. 固定 H 的状态可能丢失长距离信息；不是天然拥有无限精确记忆。
2. `h_t` 依赖 `h_(t-1)`，因此同一序列的时间步很难像无递推的运算那样完全并行；这会限制长序列训练吞吐。

这是后面 **BPTT → LSTM/GRU → Seq2Seq → Attention** 的问题来源。RNN 有真正的设计价值，也有明确的结构限制。

### 8.1 最终结论

- 逐位置 MLP 改变 C 维特征，但不负责跨 T 通信。
- 展平序列再进 Dense 可以跨时间混合，却把权重形状绑到预设 T；有其他 workaround，但不如共享时间状态自然。
- RNN 使用 `h_t=f_θ(x_t,h_(t-1))` 维护压缩历史，同一参数跨时间复用，可以处理不同长度序列。
- RNN 的状态不是无损历史数据库，且时间步之间存在串行依赖。
- 本节只解决“为什么需要 recurrence”；**下一节 8.2** 正式推导 `h_t=tanh(W_x x_t+W_h h_(t-1)+b)`，从具体数字和 shapes 开始。
